# ValuePrism leakage controls

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-truth/blob/v1.1.1/notebooks/valueprism_leakage.ipynb)

## Scope and current status

The project tests whether a model encodes how a consideration bears on an action in a particular situation. ValuePrism supplies situations, considerations, and Supports or Opposes labels, but repeated or nearly repeated moral phrases can create a shortcut. This notebook covers the split and checkerboard controls for that problem.

The Llama moral-relation development test has now passed. The human-audited confirmatory test remains sealed, and rephrasing-flip prediction has not yet run. These leakage controls determine what the development result can support and what the confirmatory stage must still establish.

Across five strict-style split draws, a 30 percent controlled injection of held-out consideration exposure raises text-only within-situation paired accuracy by 7.29 percentage points. The 95 percent Student-t interval for the mean intervention effect across those five draws runs from 4.90 to 9.69 points. Restoring situation exposure raises the mean by 0.53 points.

## Contents

1. Candidate checkerboards and reciprocal endpoint
2. Algorithmic identity grouping and split lineage
3. Cross-split overlap audit
4. Controlled shortcut restoration
5. Nested sensitivity sets and human audits
6. Exact reconstruction checks

## Start here

From GitHub, click the Open in Colab badge above. In Colab, choose Runtime and Run all. DEMO verifies and presents the public aggregate measurements on CPU. FULL retrieves ValuePrism after the dataset license has been accepted, reads HF_TOKEN from Colab Secrets, rebuilds the measurements on CPU, and keeps licensed row text inside the active runtime.

| Mode | Public input | Typical resource | Output |
| --- | --- | --- | --- |
| DEMO | Aggregate result bundle | Colab CPU, minutes | Verified tables and figures |
| FULL | Licensed ValuePrism source | Colab CPU, long run | Rebuilt manifests and comparisons |

Set GEOMETRY_OUTPUT_ROOT to a mounted Google Drive directory before the setup cell when a long reconstruction must survive a Colab reset.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-truth.git'
PUBLIC_REF = 'v1.1.1'
PUBLIC_COMMIT = 'cf605a169eef6cbe24ead242e0a5a39097df4f0d'
RUN_MODE = 'DEMO'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-truth")
    if (REPO_ROOT / ".git").is_dir():
        origin = subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], check=True, text=True, capture_output=True).stdout.strip()
        if origin.rstrip("/").removesuffix(".git") != PUBLIC_REPOSITORY.rstrip("/").removesuffix(".git"):
            raise RuntimeError("The existing checkout has an unexpected origin")
        dirty = subprocess.run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=True, text=True, capture_output=True).stdout
        if dirty:
            raise RuntimeError("The existing checkout contains modified or untracked files")
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", PUBLIC_COMMIT], check=True)
    elif not (REPO_ROOT / "pyproject.toml").is_file():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The Colab repository directory exists but is not a usable checkout")
        REPO_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "init"], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", PUBLIC_REPOSITORY], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    head = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip()
    if head != PUBLIC_COMMIT:
        raise RuntimeError(f"Expected public commit {PUBLIC_COMMIT}, found {head}")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

if RUN_MODE == "ANALYSIS":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-analysis.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif RUN_MODE == "FULL":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-reproduction.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT)], check=True)

OUTPUT_ROOT = Path(os.environ.get("GEOMETRY_OUTPUT_ROOT", "/content/geometry-results" if IN_COLAB else str(REPO_ROOT / "geometry-results")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE, "output_root": str(OUTPUT_ROOT)})

In [ ]:
from IPython.display import display

from geometry_of_truth.leakage.contracts import load_bundle, number_lineage
from geometry_of_truth.leakage.plots import stress_test
from geometry_of_truth.leakage.results import (
    audit,
    candidate_supply,
    normalization,
    overlap_checks,
    protections,
    split_lineage,
    stress_draws,
    sensitivity,
    uncertainty,
)

bundle = load_bundle(REPO_ROOT)
results = bundle["results"]
print("Artifact integrity verified")

## Candidate structure

A ValuePrism row pairs one situation with one consideration and labels that consideration Supports or Opposes. Removing the third Either label leaves 183,023 eligible rows. Among them, 20,032 situations contain at least one row of each valence, and 3,437 exact consideration phrases appear with both valences somewhere in the dataset.

Combining those reversals produces 13,923 possible checkerboards across 6,073 distinct consideration pairs. A checkerboard contains two situations and two considerations. The first consideration supports in one situation and opposes in the other, while the second consideration follows the opposite pattern. The 13,923 figure counts possible boards before semantic review, and the 6,073 figure counts unique pairs even when one pair supports several boards.

A synthetic example makes the reciprocal structure concrete.

| Synthetic unit | Academic prize | Emergency housing |
| --- | --- | --- |
| Action | Award the highest-scoring applicant | Prioritize the lowest-income applicant |
| Rewarding demonstrated merit | Supports | Opposes |
| Prioritizing urgent need | Opposes | Supports |

The labels reverse because the action and allocation purpose change, while each consideration keeps the same meaning and stakeholder role. Human review applies that same requirement to real candidates.

For scores f, the checkerboard interaction is

I = [f(s1,c1) - f(s1,c2)] + [f(s2,c2) - f(s2,c1)]

Any score formed by adding a situation term to a fixed consideration term gives I equal to zero. A nonzero reciprocal effect therefore requires sensitivity to the situation and consideration together. Nonlinear text interactions can still produce a signal, and the statistic alone cannot establish moral understanding. Text baselines and human semantic review remain necessary.

In [ ]:
display(audit(results))
display(protections(results))

## Algorithmic identity grouping

Exact string matching misses spelling and wording variants. The grouping pipeline starts with 17,678 distinct raw forms. Case, punctuation, spacing, leading articles, and light plural normalization merge 1,971 variants and leave 15,707 forms. Removing standard Value, Right, and Duty prefixes merges another 1,106 and leaves 14,601 forms.

The final automatic stage represents each phrase with character fragments of length 3 through 5 and groups forms whose similarity reaches 0.85, with at most 25 forms in one cluster. It merges 4,410 additional forms and leaves 10,191 algorithmic identity clusters. The table column named collapsed from prior reports forms merged at that stage rather than dataset rows removed.

L3 is an automatic near-duplicate grouping. Human review decides whether borderline phrases express the same consideration in meaning and stakeholder role.

In [ ]:
display(normalization(results))

## Dataset lineage

Two branches begin from the same 183,023 binary rows and answer different questions. The audit branch keeps 164,279 rows attached to consideration identities that reverse somewhere, then keeps 110,656 rows in situations containing both labels. Those rows cover 19,068 situations and measure whether enough reciprocal structure exists for review.

The frozen split branch removes 421 identical rows from the original binary pool and leaves 182,602. It holds out 25 percent of the L3 clusters and 30 percent of situations. Their joint assignment produces 116,000 strict training rows and 7,394 strict test rows, while the remaining 59,208 rows touch the held-out side for only one of the two required identities and stay outside both strict partitions.

This branching explains why 116,000 training rows plus later categories can exceed the 110,656-row audit pool. The figures come from separate filters applied to the shared source rather than successive steps in one shrinking table.

In [ ]:
display(split_lineage(results))

## Cross-split overlap audit

The strict manifests have zero row, exact-consideration, L2 prefix-stripped, L3 cluster, situation overlap. They contain two L1-normalized collisions. L1 collapses case, punctuation, spacing, articles, and light plural variants, so those two cases require a visible disposition even though the L3 cluster count is zero.

Both L1 collisions are conservatively removed from U1. The public aggregate reports their count and disposition without redistributing licensed phrase text. The accurate split claim is therefore zero L3-cluster and situation overlap, plus two disclosed L1 collisions removed from the stricter sensitivity set.

In [ ]:
display(overlap_checks(results))

## Deliberate shortcut restoration

The stress test fits the same logistic text classifier five times using seeds 0 through 4, with word unigrams and bigrams from the situation and consideration together as its features. Within-situation paired accuracy asks whether a Supports row receives a higher score than an Opposes row from the same situation, with ties worth one half. A score of 0.5 is chance and 1.0 is perfect ordering.

Each draw uses a strict-style double holdout with 25 percent of consideration clusters and 35 percent of situations held out. The final frozen manifest uses the same cluster fraction and a 30 percent situation holdout. Within each stress draw, an intervention replaces 30 percent of the capped training sample with rows exposing held-out consideration identities or held-out situations. Training size, test membership, classifier, features, and metric remain fixed.

The 30 percent consideration injection raises paired accuracy by 7.29 points on average. The sample standard deviation is 1.93 points, the standard error is 0.86, and the 95 percent Student-t interval for the mean intervention effect across these five seeds runs from 4.90 to 9.69. The situation injection raises the mean by 0.53 points, with an interval from -0.81 to 1.86. This interval covers variation across five chosen split draws. Residual duplicate exposure, label noise, human-audit uncertainty, other models, and untested ValuePrism splits remain outside its scope.

The plotted lines connect strict and restored scores for the same seed. This result belongs to the text-only pair_text baseline and within-situation paired accuracy. Activation probes and checkerboard interactions use separate endpoints.

In [ ]:
display(stress_draws(results))
display(uncertainty(results))
display(stress_test(results))

## Stricter sensitivity sets

U0 is the 7,394-row strict test set. It contains 4,372 situations, 2,009 within-situation comparisons, and 378,506 cross-situation row pairs that share a consideration cluster. U1 removes the clearest remaining near-duplicate risks and retains 7,081 rows, 4,237 situations, 1,865 within-situation comparisons, and 374,160 within-consideration pairs in total.

U2 also removes every automatically ambiguous high-risk cluster. The set collapses to 587 rows in 547 situations, with 23 within-situation comparisons and 477 within-consideration pairs. U3 is currently identical to U2 because zero human adjudications were available to construct a distinct human-confirmed layer. The terminal disposition is ULTRACLEAN_INCONCLUSIVE.

The U2 collapse shows where automatic exclusion loses the comparison structure needed for the relation test. The large within-consideration counts in U0 and U1 arise because every eligible row under one consideration can pair with rows from many situations. Human adjudication is the next filter for borderline identity pairs.

In [ ]:
sensitivity_table = sensitivity(results)
display(sensitivity_table.reset_index(drop=True))

## Two human audits

The leakage audit asks whether exposure to a training phrase substantially defeats the intended holdout of a test phrase because both express the same consideration across splits. Reviewers see phrase pairs without labels, activation results, probe scores, or baseline performance.

The checkerboard audit examines a different unit. Reviewers decide whether each consideration keeps the same meaning and stakeholder role across both situations, then check whether all four labels support the reciprocal pattern. Separate records preserve the distinction between train-test identity control and checkerboard construct validity.

## Checkerboard supply

The ranked pool contains exactly 1,090 candidate checkerboards, and the planned endpoint needs 800 human-confirmed boards. Dividing 800 by 1,090 gives a required acceptance fraction of 0.733945, or 73.3945 percent.

The 1,090 figure is an exact census of the ranked pool. Sampling uncertainty enters through the unknown human acceptance rate. Two independent reviewers first estimate that rate on a sample, and the resulting interval determines whether the pool can plausibly reach 800 accepted boards before the full audit proceeds.

In [ ]:
display(candidate_supply(results))

## Where every headline number comes from

Every displayed result comes from results.json after its integrity check passes. The final two columns below identify the exact field or calculation for each headline quantity. The fixed seeds, per-draw scores, split counts, sensitivity coverage, and candidate supply all remain available in that same public aggregate.

In [ ]:
display(number_lineage(bundle))

## Full reconstruction

FULL installs the ValuePrism dependencies, reads HF_TOKEN from Colab Secrets, and reconstructs the public aggregate on CPU. The visible comparison checks aggregate counts, all five seed-specific strict scores and intervention effects, every overlap count including L1, strict and common-training row hashes, frozen input file hashes, confirmatory row membership across all four board cells, and the U1 exclusion hash. The normalization code and counts are reproduced. The retained confirmatory hash covers unique row membership after sorting rather than candidate order. A separate public hash for the complete form-to-cluster mapping remains unavailable. ValuePrism dependency ranges also remain broader than the version-pinned Truth requirements, so a fresh FULL validation matrix is still required before claiming environment-complete reproduction.

In [ ]:
if RUN_MODE == "FULL":

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{REPO_ROOT}[valueprism-full]"], check=True)
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Add HF_TOKEN to Colab secrets and enable notebook access")
    from geometry_of_truth.leakage.reproduce import reproduce; full_run = reproduce(OUTPUT_ROOT / "valueprism-reproduction"); display(full_run["comparison"])
else:
    print("Set RUN_MODE to FULL and rerun from the first cell for raw reconstruction")

## Place in the larger experiment

The strict split measures generalization to unseen consideration identities and unseen situations. The reciprocal checkerboard measures whether a score changes with the relation after fixed consideration preferences cancel. The completed Llama development result passed its relation controls against situation-only, consideration-only, additive, and matched-text alternatives.

Human review now determines whether that development signal survives a semantically audited confirmatory set. A separate rephrasing stage will test whether the original activation geometry predicts answer flips beyond matched text and native confidence. Those open stages set the current claim boundary.